# キャリア選択と生涯所得の最適化

### F1 チーム研究 発表

**チーム名**：TODO　／　**メンバー**：TODO

> 架空のデータによる教育用シミュレーションです。実際の賃金統計ではなく、将来の所得の予測でもありません。

## この Notebook の使い方（発表前に読む・発表では表示しない）

- 発表用のスライドの**テンプレート**です。グラフは既存の `.py` から**その場で計算して**描きます。
- **TODO** の部分を、チームの研究内容・結果・考察に書き換えてください。研究で使った条件に合わせて、各セルの `with_changes` などを変更します。
- 発表のしかた（詳しくは `notebooks/README.md`）
  1. **Colab / Jupyter でそのまま**：上から実行し、見出しごとにスクロールして説明する。コードのセルは折りたたまれています。
  2. **スライドにする**：`jupyter nbconvert notebooks/04_presentation.ipynb --to slides --no-input --execute`（reveal.js の HTML スライド）
  3. **PowerPoint / Google スライドに貼る**：下の `SAVE_FIGURES = True` にして実行すると、`notebooks/figures/` に PNG が保存される

## 準備：実験環境（最初に必ず実行）

F1 リポジトリの `.py`（`scenarios.py`・`income_analysis.py`・`optimization.py` など）を import できるように、作業ディレクトリ（F1 のルート）を自動で探して設定します。

| 実行場所 | 準備 |
| --- | --- |
| ローカルの Jupyter / VSCode | `notebooks/` の中で開けば、そのまま動きます |
| Colab（方法A：GitHub から clone） | 下のセルの `REPO_URL` に**自分たちのリポジトリの URL** を入れる |
| Colab（方法B：ZIP をアップロード） | F1 フォルダを ZIP にして `/content` にアップロードしておく（またはセルの実行中にアップロード） |

> `textbook/`（教科書のコード）は公開リポジトリに含めない約束です。`textbook/` がないと `optimization.py` の import が失敗します。

In [ ]:
#@title 準備（F1 の .py を読み込む）
import os, sys, glob, zipfile, subprocess
from pathlib import Path

# ---- 方法A：GitHub から clone する場合だけ設定する（URL は自分たちのものを入れる）----
REPO_URL = ""                                  # 例："https://github.com/<team>/<repo>.git"（空なら clone しない）
REPO_SUBDIR = "f1-career-income-optimization"  # リポジトリの中の F1 フォルダ（ルートが F1 なら ""）

IN_COLAB = "google.colab" in sys.modules


def is_f1_root(p):
    return (Path(p) / "career_model.py").exists() and (Path(p) / "optimization.py").exists()


def find_f1_root():
    for p in [Path.cwd(), Path.cwd().parent]:          # ローカル：notebooks/ の中、または F1 のルート
        if is_f1_root(p):
            return p.resolve()
    if not IN_COLAB:
        return None
    if REPO_URL:                                        # Colab 方法A
        dest = Path("/content/repo")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
        cand = dest / REPO_SUBDIR if REPO_SUBDIR else dest
        if is_f1_root(cand):
            return cand
    zips = glob.glob("/content/*.zip")                  # Colab 方法B
    if not zips:
        from google.colab import files
        print("F1 フォルダの ZIP を選んでアップロードしてください")
        zips = ["/content/" + name for name in files.upload()]
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content/f1_upload")
    for hit in glob.glob("/content/f1_upload/**/career_model.py", recursive=True):
        if is_f1_root(Path(hit).parent):
            return Path(hit).parent
    return None


F1_ROOT = find_f1_root()
if F1_ROOT is None:
    raise RuntimeError("F1 のフォルダが見つかりません。上の説明（方法A／方法B）を確認してください。")
os.chdir(F1_ROOT)
if str(F1_ROOT) not in sys.path:
    sys.path.insert(0, str(F1_ROOT))
if not (F1_ROOT / "textbook" / "max.py").exists():
    print("⚠️ textbook/ がありません。optimization.py の import が失敗します。")

# ---- 既存の .py（計算の本体）と、発表用のグラフ ----
import matplotlib.pyplot as plt
import career_model as cm
import scenarios as sc
import income_analysis as ia
import optimization as op
import presentation_charts as pc

pc.setup_style()                      # 発表用に文字を大きめにする（pc.setup_style(1.3) でさらに大きく）
careers, settings = sc.load_scenarios()
named = {s["name"]: s for s in careers}
RATE = settings.get("discount_rate", ia.DEFAULT_DISCOUNT_RATE)

SAVE_FIGURES = False                  # True にすると、スライド用の PNG を notebooks/figures/ に保存する
FIG_DIR = "notebooks/figures"

def show(fig, name):
    """グラフを表示し、SAVE_FIGURES = True なら PNG も保存する"""
    if SAVE_FIGURES:
        print("saved:", pc.save(fig, f"{FIG_DIR}/{name}.png"))
    plt.show()

print("F1_ROOT:", F1_ROOT)
print("キャリア:", [s["label"] for s in careers], " 割引率:", RATE)

## 研究の問い

1. **RQ1**　就職と進学では、生涯所得がどう変わるか
2. **RQ2**　資格取得は生涯所得を増加させるか
3. **RQ3**　転職する年齢によって、生涯所得はどう変わるか
4. **RQ4**　生涯所得の最大化と、現在価値の最大化は同じか

**TODO**：チームが特に注目した問い・動機を 1〜2 行で

## モデルと仮定

- 年収：$\text{initial} \times (1+g)^{\,a - \text{start}}$、就職前は 0、18〜64 歳の 47 年
- 費用：進学（在学中に均等）、資格取得（取得した年）、転職（転職した年）
- 評価：生涯所得 ／ 累積純収入（＝所得 − 費用、**手取りではない**）／ NPV（割引率 3%）
- **単純化**：税・社会保険・インフレ・失業は扱わない。資格は必ず合格

**TODO**：チームで変更した仮定があれば追記

In [ ]:
#@title シナリオの一覧
print(f"{'Scenario':16}{'Start':>6}{'Initial':>9}{'Growth':>8}{'Costs':>8}  Notes")
for s in careers:
    notes = []
    if s.get("certificate"):
        c = s["certificate"]; notes.append(f"cert at {c['age']} (+{c['salary_increase']}/yr)")
    if s.get("job_change"):
        j = s["job_change"]; notes.append(f"switch at {j['age']} -> {j['new_salary']}")
    print(f"{s['label']:16}{s['start_age']:>6}{s['initial_salary']:>9}{s['annual_growth_rate']:>8.1%}"
          f"{ia.career_costs(s):>8,.0f}  {', '.join(notes)}")
print("（万円・歳。すべて架空の値）")

## 方法：アルゴリズム

| 処理 | アルゴリズム | 計算量 |
| --- | --- | --- |
| 年齢別所得・累積 | 配列とループ（累積和） | $O(T)$ |
| 複数キャリアの比較 | それぞれ計算 | $O(NT)$ |
| 最適な転職年齢 | **全探索** ＋ 最大値探索（教科書 `max_of`・`seq_search`） | $O(KT)$ |
| 順位付け | 安定なソート（教科書 `insertion_sort`） | $O(N^2)$ |
| 感度分析 | 仮定を 1 つずつ変えて再計算 | $O(\text{条件数} \times T)$ |

## 結果 1：年齢別の年収（RQ1）

In [ ]:
#@title グラフを表示（コードは隠しています）
show(pc.plot_income_by_age(careers), "p1_income_by_age")

**TODO**：このグラフから言えることを 1 行で

## 結果 2：累積純収入と投資回収（RQ1・RQ2）

In [ ]:
#@title グラフを表示（コードは隠しています）
show(pc.plot_cumulative_net(careers, baseline=named["early_job"]), "p2_cumulative_net")

**TODO**：何歳で追いつくか、追いつかないキャリアはあるか

## 結果 3：生涯所得と NPV の順位（RQ4）

In [ ]:
#@title グラフを表示（コードは隠しています）
results = [ia.evaluate(s, RATE) for s in careers]
fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))
for ax, key, title in [(axes[0], "gross", "Lifetime gross income"), (axes[1], "npv", f"NPV (r = {RATE:.0%})")]:
    ranked = op.rank(results, key)[::-1]
    bars = ax.barh([r["label"] for r in ranked], [r[key] for r in ranked],
                   color=[pc.color_of(r["name"]) for r in ranked], height=0.6)
    for b, r in zip(bars, ranked):
        ax.text(b.get_width(), b.get_y() + b.get_height() / 2, f" {r[key]:,.0f}", va="center", color=pc.TEXT)
    ax.set_title(title); ax.set_xlabel("[10k JPY]"); ax.grid(axis="y", visible=False)
    ax.set_xlim(0, max(r[key] for r in ranked) * 1.18)
fig.tight_layout()
show(fig, "p3_gross_vs_npv")

## 結果 4：割引率で順位が入れ替わる（RQ4）

In [ ]:
#@title グラフを表示（コードは隠しています）
show(pc.plot_rank_by_rate(careers, [0.0, 0.01, 0.02, 0.03, 0.04, 0.05]), "p4_rank_by_rate")

**TODO**：なぜ入れ替わるのか

## 結果 5：最適な転職年齢（RQ3）

In [ ]:
#@title グラフを表示（コードは隠しています）
jc = named["job_change"]          # TODO：研究で使った条件に変える（例：sc.with_changes(jc, **{"job_change.new_salary": 380})）
first, last = settings.get("switch_age_range", [22, 60])
rows = op.evaluate_switch_ages(jc, op.switch_age_candidates(jc, first, last), RATE)
best = op.optimal_switch_ages(rows)
stay = ia.evaluate(sc.with_changes(jc, job_change=None), RATE)
show(pc.plot_switch_age_search(rows, best, stay, f"Best job-change age ({len(rows)} candidates, O(KT))"),
     "p5_switch_search")
print("最適な転職年齢：" + "、".join(f"{op.CRITERIA[k]} {v['switch_age']} 歳" for k, v in best.items()))

**TODO**：結果を決めている仮定は何か（例：転職後の年収が経験によらない）

## 結果 6：感度分析

In [ ]:
#@title グラフを表示（コードは隠しています）
base, uni = named["early_job"], named["university"]     # TODO：比べる 2 つのキャリアを選ぶ
diff = lambda s: ia.cumulative_net_income(s) - ia.cumulative_net_income(base)
sens = []
for key, label in [("initial_salary", "initial salary"), ("annual_growth_rate", "growth rate"),
                   ("education_cost", "education cost")]:
    sens.append({"label": f"{label} ±20%", "low": diff(sc.with_changes(uni, **{key: uni[key] * 0.8})),
                 "high": diff(sc.with_changes(uni, **{key: uni[key] * 1.2}))})
show(pc.plot_tornado(sens, diff(uni), "What drives University minus Early Job?", "difference [10k JPY]"),
     "p6_tornado")

## 発展研究（Advanced）

**TODO**：選んだ Advanced A〜E のテーマ・方法・結果（グラフはこの下にセルを追加）

- A：モンテカルロ　B：動的計画法　C：所得以外の評価　D：公的統計　E：リスクと最適化

## 結論

- **RQ1**：TODO
- **RQ2**：TODO
- **RQ3**：TODO
- **RQ4**：TODO

**どの仮定の下での結論か**を必ず添えてください。

## モデルの限界と注意

- 架空のデータによる教育用シミュレーション。将来の所得の予測・保証ではない
- 額面の所得であり、手取り（可処分所得）ではない。税・社会保険・インフレ・失業は扱っていない
- 学歴・資格・転職の効果は人や時代によって違う。統計の差を因果効果と同一視しない
- **所得だけで人生やキャリアの価値は決まらない**
- 性別・国籍・出身地などによる差別的な判断を推奨するものではない

**TODO**：自分たちの研究で特に気をつけた点

## 参考文献

- 柴田望洋『新・明解 Pythonで学ぶアルゴリズムとデータ構造（第2版）』SBクリエイティブ
- F1 リポジトリ `README.md`、データ `data/sample_careers.json`（架空）
- **TODO**：チームで使った文献・統計（調査年・対象・単位・URL・参照日）

### ご清聴ありがとうございました